# Hybrid Mechanistic-ML Bioreactor Digital Twin (`E. coli BL21` Fed-Batch)

Reproducible Portela Level 3+ workflow implementing:
1. **Stage 1**: Multi-batch iDoE mechanistic kinetic recovery (`mu_max, K_S, Y_XS, alpha, beta, k_La0`) with relative FIM conditioning (`kappa_rel`).
2. **Stage 2**: Zero-`X_d`-leakage `GAM-86` structured serial hybrid Neural ODE trained via reverse-mode Exponential-Rosenbrock VJP + `B=15` bootstrap ensemble (`Pinto et al., 2019`) + subspace Mahalanobis Domain of Validity (DoV).
3. **Intensive 1000 L Geometric Scale-Up**: Zero extensive volume inputs (`Pg/V, v_s, v_tip`).
4. **Sensor-Parity Hybrid EKF**: Equal base `Q_DEFAULT` & sensors with exact `expm(F*dt)` covariance and `7x5` stoichiometric `Q_epistemic` propagation.
5. **Portela Level 3 `BioreactorDigitalTwin`**: Delayed at-line LIMS replay & ALCOA+ SHA-256 provenance.

In [1]:
import numpy as np
from hybrid_fedbatch_twin import (
    BioreactorParams,
    OperatingInputs,
    BioreactorDigitalTwin,
    ClosedLoopFeedController,
    compute_hydrodynamics,
    compute_rates,
    default_initial_state,
    recover_mechanistic_parameters,
    train_default_hybrid_model,
    benchmark_controllers,
)

true_params = BioreactorParams()
recovery = recover_mechanistic_parameters(noise_std=0.015, seed=42, true_params=true_params)
params_rec = recovery["recovered_params"]

print("Stage-1 Multi-Batch iDoE Recovery:")
for p_name in recovery["param_names"]:
    print(
        f"  {p_name:<7} | true={recovery['true_values'][p_name]:8.4f} "
        f"| rec={recovery['recovered_values'][p_name]:8.4f} "
        f"| SE={recovery['standard_errors'][p_name]:7.4f} "
        f"| err={recovery['relative_error_pct'][p_name]:5.2f}%"
    )
print(
    f"Max RelErr: {recovery['max_relative_error_pct']:.2f}% | "
    f"Rel FIM kappa: {recovery['condition_number_rel']:.2f} | "
    f"SHA256: {recovery['calibration_sha256'][:12]}"
)

Stage-1 Multi-Batch iDoE Recovery:
  mu_max  | true=  0.4500 | rec=  0.4587 | SE= 0.0037 | err= 1.93%
  K_S     | true=  0.1500 | rec=  0.1557 | SE= 0.0029 | err= 3.83%
  Y_XS    | true=  0.4800 | rec=  0.4757 | SE= 0.0016 | err= 0.89%
  alpha   | true=  0.3500 | rec=  0.3462 | SE= 0.0038 | err= 1.09%
  beta    | true=  0.0250 | rec=  0.0257 | SE= 0.0005 | err= 2.91%
Max RelErr: 3.83% | Rel FIM kappa: 11.56 | SHA256: dbd0634285ed


In [2]:
model, metrics = train_default_hybrid_model(
    params=params_rec,
    plant_params=true_params,
    noise_std=0.015,
    seed=7,
    n_bootstrap=15,
)
print(
    f"GAM-86 (n_params={int(metrics['n_params'])}, B={int(metrics['n_bootstrap'])}) | "
    f"AIC={metrics['aic']:.1f} | BIC={metrics['bic']:.1f} | DoV p99={metrics['dov_p99']:.2f}"
)
print(
    f"Held-out Specific Rates R2 -> "
    f"k_d={metrics['kd_r2']:.4f}, k_La={metrics['kla_r2']:.4f}, "
    f"q_A_net={metrics['qAnet_r2']:.4f}, mu={metrics['mu_r2']:.4f}"
)
print(
    f"Open-Loop Rollout nRMSE -> Hybrid={metrics['hybrid_norm_rmse']:.4f} vs "
    f"Nominal={metrics['nominal_norm_rmse']:.4f} "
    f"({100.0 * (1.0 - metrics['hybrid_norm_rmse'] / metrics['nominal_norm_rmse']):.1f}% reduction)"
)

GAM-76 (n_params=76, B=15) | AIC=-13186.4 | BIC=-12782.8 | DoV p99=3.25
Held-out Specific Rates R2 -> k_d=0.8507, k_La=0.9932, q_A_net=0.9698, mu=0.9838
Open-Loop Rollout nRMSE -> Hybrid=0.1379 vs Nominal=0.5950 (76.8% reduction)


In [3]:
from scipy.optimize import brentq

p5 = params_rec
p1000 = BioreactorParams(**{**params_rec.to_dict(), "V_vessel": 1000.0, "V_0": 400.0, "V_max": 1000.0})
rpm5, vvm5, v5 = 520.0, 1.25, 3.5
hyd5 = compute_hydrodynamics(rpm5, vvm5, v5, p5)
v1000 = v5 * (1000.0 / 5.0)
vvm1000 = vvm5 * (5.0 / 1000.0) ** (1.0 / 3.0)
rpm1000 = brentq(
    lambda r: compute_hydrodynamics(r, vvm1000, v1000, p1000)["Pg_V"] - hyd5["Pg_V"],
    40.0,
    400.0,
)
x5 = np.array([12.0, 0.8, 0.35, 3.2, 0.45, 2.5, v5])
x1000 = x5.copy(); x1000[6] = v1000
r5 = compute_rates(x5, OperatingInputs(F=0.12, rpm=rpm5, vvm=vvm5), p5, mode="hybrid", residual_model=model)
r1000 = compute_rates(x1000, OperatingInputs(F=24.0, rpm=float(rpm1000), vvm=vvm1000), p1000, mode="hybrid", residual_model=model)
print(
    f"5 L -> 1000 L Matched (Pg/V, v_s): |d Pg_V|={abs(r5['Pg_V']-r1000['Pg_V']):.2e}, "
    f"|d k_La|={abs(r5['k_La']-r1000['k_La']):.2e} | v_tip: {r5['v_tip']:.3f} -> {r1000['v_tip']:.3f} m/s"
)

# Sensor-parity 3-way closed-loop control & EKF benchmark
bench = benchmark_controllers(
    params=params_rec,
    residual_model=model,
    duration_h=16.0,
    mu_target=0.28,
    rpm=450.0,
    vvm=1.2,
    plant_params=true_params,
)
for name, entry in bench.items():
    s = entry["summary"]
    print(
        f"{name:<20} | Xv_end={s['final_Xv_g_L']:5.2f} | A_peak={s['peak_acetate_g_L']:5.3f} "
        f"| minDO%={s['min_DO_pct']:4.1f}% | Hyb_nRMSE={s['hybrid_norm_rmse']:.4f} "
        f"| Nom_nRMSE={s['nominal_norm_rmse']:.4f} | max_Dm={s['max_d_m']:.2f}"
    )

5 L -> 1000 L Matched (Pg/V, v_s): |d Pg_V|=5.68e-14, |d k_La|=1.14e-13 | v_tip: 1.402 -> 2.438 m/s
hybrid_closed_loop   | Xv_end=25.12 | A_peak=0.126 | minDO%=22.5% | Hyb_nRMSE=0.0117 | Nom_nRMSE=0.0702 | max_Dm=2.38
nominal_closed_loop  | Xv_end=21.51 | A_peak=4.764 | minDO%= 2.2% | Hyb_nRMSE=0.1124 | Nom_nRMSE=0.9539 | max_Dm=3.44
open_loop_exp        | Xv_end=20.25 | A_peak=2.815 | minDO%= 6.2% | Hyb_nRMSE=0.0592 | Nom_nRMSE=0.6421 | max_Dm=4.23


In [4]:
# Portela Level 3 Closed-Loop Session with Delayed LIMS Replay & ALCOA+ Provenance
twin = BioreactorDigitalTwin(
    x0=default_initial_state(params_rec),
    params=params_rec,
    residual_model=model,
    calibration_metadata={"calibration_sha256": recovery["calibration_sha256"]},
)
ctrl = ClosedLoopFeedController(params=params_rec, residual_model=model)
for step in range(1, 11):
    th = 0.1 * step
    act = twin.compute_control_action(th, ctrl)
    twin.ingest_online_pat(th, 1.5 + 0.06 * step, 7.1, 2.0, act["F_cmd"], act["rpm_cmd"], act["vvm_cmd"])

replay = twin.ingest_atline_assay(t_sample_h=0.4, y_S=0.30, y_A=0.55, y_P=0.20)
audit = twin.to_audit_record()
print(
    f"Delayed LIMS replay complete (t_sample=0.4h -> t_now={replay['t_now']:.1f}h, "
    f"replayed_steps={replay['replayed_steps']}, mu_hat={replay['mu_hat']:.4f}, D_M={replay['d_m']:.2f})"
)
print(
    f"ALCOA+ Audit Record: events={audit['audit_log_count']}, "
    f"params_sha={audit['params_sha256'][:12]}, "
    f"model_sha={audit['model_sha256'][:12]}, "
    f"record_sha={audit['record_sha256'][:12]}"
)

Delayed LIMS replay complete (t_sample=0.4h -> t_now=1.0h, replayed_steps=6, mu_hat=0.2371, D_M=1.50)
ALCOA+ Audit Record: events=22, params_sha=dbd0634285ed, model_sha=cff016e546b3, record_sha=8e77221a86d7
